## 05 - Word Frequency Computing

This notebook computes a word frequency distribution for our dataset.

>**INPUTS**:

- **Source dataset**: `interventions_filtered.csv` (processed and filtered interventions)
- **NLP model**: spaCy Spanish model `es_core_news_lg`

> **OUTPUTS** :
- **Frequency distributions** organized in a dictionaryas:  
    `(Political Group, Legislative Year, Word) → Count`

In [ ]:
import pandas as pd
import numpy as np
import sys
from collections import Counter
from pathlib import Path
import spacy
import pickle

DATA_DIR = Path('..') / "data" / "processed"
OUTPUT_DIR = Path('..') / "data" / "processed"

OUTPUT_DIR.mkdir(exist_ok=True)

nlp = spacy.load("es_core_news_lg")


In [ ]:
data_path = DATA_DIR / "interventions_filtered.csv"

df = pd.read_csv(data_path)

df.head()

### Data Organization

We organize political groups and time periods as follows:

**Political Groupings**:
The dataset includes four major political groups analyzed separately and combined:
- PP (Prominent center-right party)
- PSOE (Promintent center-left party)  
- IU-PODEMOS (United Left-Podemos coalition)
- VOX (Spanish far-right party)

We further stratify by gender (Male/Female).

**Temporal Coverage**:
Analysis covers all legislative years from 1979 to 2024.

In [ ]:
POLITICAL_GROUPS = ["VOX", "PP", "PSOE", "IU-PODEMOS"]
GENDERS = ["Male", "Female"]

# Create aggregate indices: overall groups + gender-stratified groups
indices = POLITICAL_GROUPS.copy()

for group in POLITICAL_GROUPS:
    for gender in GENDERS:
        indices.append(f"{group} - {gender}")

print(f"Political groups: {POLITICAL_GROUPS}")
print(f"Total indices (groups + gender combinations): {len(indices)}")

# Generate year range
MIN_YEAR = int(df["year"].min())
MAX_YEAR = int(df["year"].max())
YEARS = list(range(MIN_YEAR, MAX_YEAR + 1))

print(f"Temporal range: {YEARS[0]} to {YEARS[-1]}")

In [ ]:
# Test tokenization on a sample text
sample_idx = 23
sample_text = df.loc[sample_idx, "text"].lower()

print(f"\nSample text (first 200 chars):\n{sample_text[:200]}...\n")

doc = nlp(sample_text)
tokens = [token.text for token in doc 
          if not token.is_punct and not token.is_space]

print(f"Total tokens (after filtering punctuation): {len(tokens)}")
print(f"Unique tokens: {len(set(tokens))}")
print(f"Sample tokens: {tokens[:20]}")

### Frequency Distribution Computation

The following cells compute word frequencies organized by:
- Political group
- Gender (where applicable)
- Legislative year

For each intervention, we:
1. Tokenize and lemmatize the text using spaCy
2. Filter out punctuation and whitespace
3. Aggregate tokens by group and year

We'll get a dictionary with the following structure:

    `freq[<year>]['<group>']['<word>'] -> "word frequency by given group on given year"`
    `freq[<year>]['<group> - <gender>']['<word>'] -> "word frequency by given group (just of a specific gender) on given year"`
    `freq[<year>]['<group>']['TOTAL_WORDS'] -> "total word frequency by given group on given year"`

In [ ]:
# Start storing tokens in a dictionary (year -> index -> list of tokens)
word_counts = {year: {idx: [] for idx in indices} for year in YEARS}

for i, row in df.iterrows():
    if i % 10000 == 0:
        print(f"Processed {i}/{len(df)} interventions ({100*i/len(df):.1f}%)")

    try:
        text = str(row["text"]).lower()
        year = int(row["year"])
        group = row["group"]
        gender = row["gender"]

        # Skip unexpected records
        if year not in word_counts or group not in word_counts[year]:
            continue

        doc = nlp(text)
        tokens = [t.text for t in doc if not t.is_punct and not t.is_space]

        # General group
        word_counts[year][group].extend(tokens)

        # Gender-stratified group
        gender_key = f"{group} - {gender}"
        if gender_key in word_counts[year]:
            word_counts[year][gender_key].extend(tokens)

    except Exception:
        continue

print(f"Extraction completed for {len(word_counts)} years.")


In [ ]:
# Convert raw token lists to Counter objects
frequencies = {year: {idx: Counter(word_counts[year][idx]) 
                      for idx in indices} 
               for year in YEARS}

# Add total word count as a special key for each category
for year in YEARS:
    for idx in indices:
        frequencies[year][idx]["TOTAL_WORDS"] = sum(frequencies[year][idx].values())


print(f"  - Categories: {len(indices)}")
print(f"  - Years: {len(YEARS)}")

In [ ]:
# Save as pickle for efficient reloading
output_pickle = OUTPUT_DIR / "word_frequencies.pkl"
with open(output_pickle, "wb") as f:
    pickle.dump(frequencies, f)